# Impact indicators, matching, and regression models

This notebook presents the code behind the manuscript's academic, policy, technological, and clinical impact analyses. It shows how variables and matched controls were constructed and how models were specified. It does not contain the underlying publication-level data or model result tables.


## Prepare the analytical cohort and impact indicators

Read the labeled cohort, limit the publication period to 2009–2022, link normalized five-year citations, and construct indicators for citations in policy documents, patents, and clinical trials.


In [ ]:
df_0 = pd.read_json('a5_data_final_v3.json', lines=True)

In [ ]:
df_0['year'] = df_0['year'].astype(int)
df_1 = df_0[df_0['year'] >= 2009]

In [ ]:
df_ac = df_1.copy()

In [ ]:
df_ac1 = pd.read_parquet("matched_rows.parquet")
df_ac1 = df_ac1.drop_duplicates(subset=['paperid','doi'])[['PMID','paperid']]
df_merge_0 = df_ac.merge(df_ac1, left_on='PMID', right_on='PMID', how='inner')

In [ ]:
df_jl = pd.read_csv('sciscinet2\gu_data_c5_normalized_20251122.csv')
df_merged = df_merge_0.merge(df_jl, left_on='paperid', right_on='paper_id', how='inner')

In [ ]:
df_policy = df_1.copy()
df_policy['Cited_by_policy'] = (df_policy['total_sum']>0).astype(int)
df_policy['inst_type'] = df_policy['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })
df_summary_p = summarize_by_mesh(df_policy,'total_sum')
merged_df_p = df_summary_p.merge(df_mesh_lvl1, on='mesh_lvl1', how='left')
df_summary_1p = summarize_by_mesh(df_policy,'Cited_by_policy')
merged_df_1p = df_summary_1p.merge(df_mesh_lvl1, on='mesh_lvl1', how='left')
merged_df_p = merged_df_p[merged_df_p['category_name']!='Animal Diseases']
merged_df_1p = merged_df_1p[merged_df_1p['category_name']!='Animal Diseases']
df_overall_p = df_policy[['inst_type','total_sum','Cited_by_policy']].groupby('inst_type').mean().reset_index().rename(columns = {'total_sum':'mean_left','Cited_by_policy':'mean_right'})
df_overall_p['mean_right']=df_overall_p['mean_right']*100
df_overall_p['inst_type'] = df_overall_p['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })
inst_order=("public", "mixed", "non-public")
df_overall_p['inst_type'] = pd.Categorical(df_overall_p['inst_type'], categories=list(inst_order), ordered=True)
df_overall_p = df_overall_p.sort_values(['inst_type'])

In [ ]:
df_pp = pd.read_csv("PKG\C16_Link_Patents_Papers.tsv", encoding="utf-8", sep="\t")
df_ppc = df_pp.drop_duplicates(['PMID','PatentId'])[['PMID','PatentId']].groupby('PMID').count().reset_index()
df_final_pattern = df_1.merge(df_ppc, on="PMID", how="left")
df_final_pattern["PatentId"] = df_final_pattern["PatentId"].fillna(0).astype(int)
df_final_pattern["has_patent"] = (df_final_pattern["PatentId"] > 0).astype(int)
df_final_pattern['inst_type'] = df_final_pattern['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

df_summary_pp = summarize_by_mesh(df_final_pattern,'PatentId')
merged_df_pp = df_summary_pp.merge(df_mesh_lvl1, on='mesh_lvl1', how='left')
df_summary_1_pp = summarize_by_mesh(df_final_pattern,'has_patent')
merged_df_1_pp = df_summary_1_pp.merge(df_mesh_lvl1, on='mesh_lvl1', how='left')

merged_df_pp = merged_df_pp[merged_df_pp['category_name']!='Animal Diseases']
merged_df_1_pp = merged_df_1_pp[merged_df_1_pp['category_name']!='Animal Diseases']
df_overall_pp = df_final_pattern[['inst_type','PatentId','has_patent']].groupby('inst_type').mean().reset_index().rename(columns = {'PatentId':'mean_left','has_patent':'mean_right'})
df_overall_pp['mean_right']=df_overall_pp['mean_right']*100
df_overall_pp['inst_type'] = df_overall_pp['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })
inst_order=("public", "mixed", "non-public")
df_overall_pp['inst_type'] = pd.Categorical(df_overall_pp['inst_type'], categories=list(inst_order), ordered=True)
df_overall_pp = df_overall_pp.sort_values(['inst_type'])
df_overall_pp

In [ ]:
df_pc = pd.read_csv("PKG\C12_Link_Papers_Clinicaltrials.tsv", encoding="utf-8", sep="\t")
df_pcc = df_pc.drop_duplicates(['PMID','nct_id'])[['PMID','nct_id']].groupby('PMID').count().reset_index()
df_final_clinical = df_1.merge(df_pcc, on="PMID", how="left")
df_final_clinical["nct_id"] = df_final_clinical["nct_id"].fillna(0).astype(int)
df_final_clinical["has_CT"] = (df_final_clinical["nct_id"] > 0).astype(int)
df_final_clinical['inst_type'] = df_final_clinical['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

df_summary_c = summarize_by_mesh(df_final_clinical,'nct_id')
merged_df_c = df_summary_c.merge(df_mesh_lvl1, on='mesh_lvl1', how='left')
df_summary_c_1 = summarize_by_mesh(df_final_clinical,'has_CT')
merged_df_c_1 = df_summary_c_1.merge(df_mesh_lvl1, on='mesh_lvl1', how='left')
merged_df_c = merged_df_c[merged_df_c['category_name']!='Animal Diseases']
merged_df_c_1 = merged_df_c_1[merged_df_c_1['category_name']!='Animal Diseases']    

df_overall_c = df_final_clinical[['inst_type','nct_id','has_CT']].groupby('inst_type').mean().reset_index().rename(columns = {'nct_id':'mean_left','has_CT':'mean_right'})
df_overall_c['mean_right']=df_overall_c['mean_right']*100
df_overall_c['inst_type'] = df_overall_c['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })
inst_order=("public", "mixed", "non-public")
df_overall_c['inst_type'] = pd.Categorical(df_overall_c['inst_type'], categories=list(inst_order), ordered=True)
df_overall_c = df_overall_c.sort_values(['inst_type'])
df_overall_c

## Academic citation models

Merge normalized five-year citation information and journal measures, define highly cited papers, and specify overall and disease-specific OLS and logistic models. Academic models use the available full sample rather than CEM controls.


In [ ]:
df_jl = pd.read_csv('sciscinet2\gu_data_c5_normalized_20251122.csv')
df_ac1 = pd.read_parquet("matched_rows.parquet")
df_ac1 = df_ac1.drop_duplicates(subset=['paperid','doi'])[['PMID','paperid']]

In [ ]:
df_merge_0 = df_1.merge(df_ac1, left_on='PMID', right_on='PMID', how='inner')
df_merged = df_merge_0.merge(df_jl, left_on='paperid', right_on='paper_id', how='inner')

In [ ]:
df_citation = df_merged[['PMID', 'year', 'matched_IDs','matched_count','inst_type', 'country_type', 'country_type_v2','mesh_lvl1','team_size_group','document_type_group', 'mesh_lvl2','c5_norm']]
df_citation

In [ ]:
top1_cutoff = df_citation['c5_norm'].quantile(0.99)
top5_cutoff = df_citation['c5_norm'].quantile(0.95)
top10_cutoff = df_citation['c5_norm'].quantile(0.90)

mean_c5 = df_citation['c5_norm'].mean()


df_citation['top1_pct'] = (
    df_citation['c5_norm'] >= top1_cutoff
).astype(int)

df_citation['top5_pct'] = (
    df_citation['c5_norm'] >= top5_cutoff
).astype(int)

df_citation['top10_pct'] = (
    df_citation['c5_norm'] >= top10_cutoff
).astype(int)

df_citation['above_avg'] = (
    df_citation['c5_norm'] > mean_c5
).astype(int)

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_citation_j = pd.merge(df_citation, d_journal_2, on='PMID', how='inner')

In [ ]:
d_journali = pd.read_csv('a3_JIF.csv',encoding='utf-8').drop_duplicates()
d_journal_2 = d_journali[['id', 'journalName', 'jcrYear', 'abbrJournal',
       'category','totalArticles', 'jif2019', 'jif5Years',
       'averageJifPercantage', 'jci', 'quartile',
       'immediacyIndex','jifPercentile','title', 'yearPublished', 'jciPercentile','aisQuartile']]

In [ ]:
d_journal_3 = d_journali[['id', 'journalName', 'jcrYear','jifPercentile','quartile']].dropna(subset=['jifPercentile']).copy()
#d_journal_3['q_num'] = d_journal_3['quartile'].str.replace('Q', '').astype(int)

conditions = [
    d_journal_3['jifPercentile'] > 75,
    d_journal_3['jifPercentile'] > 50,
    d_journal_3['jifPercentile'] > 25,
    d_journal_3['jifPercentile'] >= 0
]

choices = [1, 2, 3, 4]

d_journal_3['q_num'] = np.select(
    conditions,
    choices,
    default=np.nan
).astype(int)

bins = [0,10,20,30,40,50,60,70,80,90,100]
labels = [10,9,8,7,6,5,4,3,2,1]

d_journal_3['decile'] = pd.cut(
    d_journal_3['jifPercentile'],
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

In [ ]:
d_journal_3['journal_match'] = (
    d_journal_3['journalName']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

df_citation_j['journal_match'] = (
    df_citation_j['source_title']
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# merge
df_citation_j2 = df_citation_j.merge(
    d_journal_3[['journal_match', 'jcrYear', 'jifPercentile', 'q_num', 'decile']],
    left_on=['journal_match', 'year'],
    right_on=['journal_match', 'jcrYear'],
    how='left'
)


df_citation_j2.drop(columns=['journal_match', 'jcrYear'], inplace=True)

In [ ]:
df_citation_j3 = df_citation_j2[
    df_citation_j2['jifPercentile'].notna()
].copy()

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    'c5_norm ~ C(inst_type) + C(year)+C(q_num)+C(team_size_group)+C(document_type_group)',
    data=df_citation_j3
).fit()

print(model.summary())

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    'c5_norm ~ C(inst_type, Treatment(reference="public")) + C(year)+C(q_num)+C(team_size_group)+C(document_type_group)',
    data=df_citation_j3
).fit()

print(model.summary())

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    'c5_norm ~ C(inst_type) + C(year)+C(q_num)',
    data=df_citation_j3
).fit()

print(model.summary())

In [ ]:
import statsmodels.api as sm
import patsy
formula = "top10_pct ~ C(inst_type)+ C(year)+C(q_num)+C(team_size_group)+C(document_type_group)"  

y, X = patsy.dmatrices(formula, df_citation_j3, return_type="dataframe")

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy
formula = "top10_pct ~ C(inst_type, Treatment(reference='public'))+ C(year)+C(q_num)+C(team_size_group)+C(document_type_group)"  

y, X = patsy.dmatrices(formula, df_citation_j3, return_type="dataframe")

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
)

res = model.fit()
print(res.summary())

In [ ]:
df_mesh_lvl1=pd.read_csv("mesh_lvl1_disease_mapping.csv")
mesh_list = df_mesh_lvl1['mesh_lvl1'].to_list()

In [ ]:
df_citation_j3['inst_type'] = df_citation_j3['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
def choose_disease(df,mesh_code):
    1
    return df[df['mesh_lvl1'].apply(lambda x: mesh_code in x if isinstance(x, list) else False)]

In [ ]:
df_list_citation = []

for disease in mesh_list:
    time_1 = time.time()
    df_disease = choose_disease(df_citation_j3, disease)
    df_list_citation.append([disease, df_disease])

In [ ]:
def get_disease_data_0(df_list_policy, alpha=0.05):
    all_results = []

    inst_types = ["total", "public", "non-public", "mixed"]

    for disease, df_disease, in df_list_policy:
        df_matched=df_disease
        for inst in inst_types:

            # ====== df_disease ======
            if inst == "total":
                d_sub = df_disease
            else:
                d_sub = df_disease[df_disease["inst_type"] == inst]

            d_t1 = d_sub[d_sub["top10_pct"] == 1]["PMID"].nunique()
            d_t0 = d_sub[d_sub["top10_pct"] == 0]["PMID"].nunique()

            # ====== df_matched ======
            if inst == "total":
                m_sub = df_matched
            else:
                m_sub = df_matched[df_matched["inst_type"] == inst]

            m_t1 = m_sub[m_sub["top10_pct"] == 1]["PMID"].nunique()
            m_t0 = m_sub[m_sub["top10_pct"] == 0]["PMID"].nunique()


            row = pd.DataFrame([{
                "disease": disease,
                "inst_type": inst,

                "number_treatment_before_CEM": d_t1,
                "number_control_before_CEM": d_t0,

                "number_treatment_after_CEM": m_t1,
                "number_control_after_CEM": m_t0
            }])

            all_results.append(row)

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    else:
        return pd.DataFrame()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf


def logistic_inst_type_compare(
    df,
    disease_mesh,
    journal_var='decile',
    alpha=0.05
):
    """
    Fit three regression models for one disease and return separate result tables.

    Parameters
    ----------
    df : pandas.DataFrame
        Data for the current disease.
    disease_mesh : str
        Disease MeSH code, such as 'C01'.
    journal_var : str, default='decile'
        Journal-rank control variable.
    alpha : float, default=0.05
        Significance threshold.

    Returns
    -------
    dict
        Contains three DataFrames:

        {
            'c5_norm_OLS': DataFrame,
            'top1_pct_logit': DataFrame,
            'top10_pct_logit': DataFrame
        }
    """

    df = df.copy()


    df['inst_type'] = pd.Categorical(
        df['inst_type'],
        categories=[
            'private',
            'public',
            'public+private'
        ]
    )

    models = {
        'c5_norm_OLS': {
            'outcome': 'c5_norm',
            'type': 'ols'
        },
        'top1_pct_logit': {
            'outcome': 'top1_pct',
            'type': 'logit'
        },
        'top10_pct_logit': {
            'outcome': 'top10_pct',
            'type': 'logit'
        }
    }


    output_tables = {
        model_name: []
        for model_name in models
    }

    coefficient_names = {
        'public': (
            'C(inst_type, '
            'Treatment(reference="private"))[T.public]'
        ),
        'mixed': (
            'C(inst_type, '
            'Treatment(reference="private"))'
            '[T.public+private]'
        )
    }

    comparison_labels = {
        'public': 'public vs non-public',
        'mixed': 'mixed vs non-public'
    }

    for model_name, spec in models.items():

        outcome = spec['outcome']

        try:
            model_df = df.dropna(
                subset=[
                    outcome,
                    'inst_type',
                    'year',
                    journal_var
                ]
            ).copy()

            if model_df.empty:
                raise ValueError(
                    'No valid observations after dropping NA'
                )

            if model_df['inst_type'].nunique() < 2:
                raise ValueError(
                    'Less than two inst_type groups'
                )

            if (
                spec['type'] == 'logit'
                and model_df[outcome].nunique() < 2
            ):
                raise ValueError(
                    f'{outcome} has only one value'
                )

            formula = (
                f'{outcome} ~ '
                'C(inst_type, Treatment(reference="private")) '
                f'+ C(year) + C({journal_var})+C(team_size_group)+C(document_type_group)'
            )

            if spec['type'] == 'ols':
                fitted_model = smf.ols(
                    formula=formula,
                    data=model_df
                ).fit()

            else:
                fitted_model = smf.glm(
                    formula=formula,
                    data=model_df,
                    family=sm.families.Binomial()
                ).fit()


            for group in ['public', 'mixed']:

                parameter_name = coefficient_names[group]

                coef = fitted_model.params.get(
                    parameter_name,
                    np.nan
                )

                p_value = fitted_model.pvalues.get(
                    parameter_name,
                    np.nan
                )

                significant = (
                    bool(p_value < alpha)
                    if pd.notna(p_value)
                    else False
                )

                if significant and coef > 0:
                    larger = group
                    smaller = 'non-public'

                elif significant and coef < 0:
                    larger = 'non-public'
                    smaller = group

                else:
                    larger = ''
                    smaller = ''

                output_tables[model_name].append({
                    'Disease Mesh': disease_mesh,
                    'Comparison Model':
                        comparison_labels[group],
                    'coef': coef,
                    'p_value': p_value,
                    'significant': significant,
                    'Larger if significant': larger,
                    'Smaller if significant': smaller
                })

        except Exception as error:


            for group in ['public', 'mixed']:
                output_tables[model_name].append({
                    'Disease Mesh': disease_mesh,
                    'Comparison Model':
                        comparison_labels[group],
                    'coef': np.nan,
                    'p_value': np.nan,
                    'significant': False,
                    'Larger if significant': '',
                    'Smaller if significant': '',
                    'error': str(error)
                })


    output_tables = {
        model_name: pd.DataFrame(rows)
        for model_name, rows in output_tables.items()
    }

    return output_tables

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

def logistic_inst_type_compare(df, disease_mesh, journal_var='decile', alpha=0.05):
    df = df.copy()
    df['inst_type'] = pd.Categorical(df['inst_type'], categories=['private', 'public', 'public+private'])

    models = {
        'c5_norm_OLS': {'outcome': 'c5_norm', 'type': 'ols'},
        'top1_pct_logit': {'outcome': 'top1_pct', 'type': 'logit'},
        'top10_pct_logit': {'outcome': 'top10_pct', 'type': 'logit'}
    }

    output_tables = {model_name: [] for model_name in models}

    public_name = 'C(inst_type, Treatment(reference="private"))[T.public]'
    mixed_name = 'C(inst_type, Treatment(reference="private"))[T.public+private]'

    for model_name, spec in models.items():
        outcome = spec['outcome']

        try:
            model_df = df.dropna(subset=[outcome, 'inst_type', 'year', journal_var, 'team_size_group', 'document_type_group']).copy()

            if model_df.empty:
                raise ValueError('No valid observations after dropping NA')
            if model_df['inst_type'].nunique() < 2:
                raise ValueError('Less than two inst_type groups')
            if spec['type'] == 'logit' and model_df[outcome].nunique() < 2:
                raise ValueError(f'{outcome} has only one value')

            formula = f'{outcome} ~ C(inst_type, Treatment(reference="private")) + C(year) + C({journal_var}) + C(team_size_group) + C(document_type_group)'

            if spec['type'] == 'ols':
                fitted_model = smf.ols(formula=formula, data=model_df).fit()
            else:
                fitted_model = smf.glm(formula=formula, data=model_df, family=sm.families.Binomial()).fit()

            comparisons = []

            for label, param, group1, group2 in [
                ('public vs non-public', public_name, 'public', 'non-public'),
                ('mixed vs non-public', mixed_name, 'mixed', 'non-public')
            ]:
                coef = fitted_model.params.get(param, np.nan)
                p_value = fitted_model.pvalues.get(param, np.nan)
                comparisons.append((label, coef, p_value, group1, group2))

            if public_name in fitted_model.params.index and mixed_name in fitted_model.params.index:
                contrast = np.zeros(len(fitted_model.params))
                contrast[fitted_model.params.index.get_loc(mixed_name)] = 1
                contrast[fitted_model.params.index.get_loc(public_name)] = -1
                test = fitted_model.t_test(contrast)
                coef = float(np.asarray(test.effect).squeeze())
                p_value = float(np.asarray(test.pvalue).squeeze())
            else:
                coef, p_value = np.nan, np.nan

            comparisons.append(('mixed vs public', coef, p_value, 'mixed', 'public'))

            for label, coef, p_value, group1, group2 in comparisons:
                significant = bool(p_value < alpha) if pd.notna(p_value) else False

                if significant and coef > 0:
                    larger, smaller = group1, group2
                elif significant and coef < 0:
                    larger, smaller = group2, group1
                else:
                    larger, smaller = '', ''

                output_tables[model_name].append({
                    'Disease Mesh': disease_mesh,
                    'Comparison Model': label,
                    'coef': coef,
                    'p_value': p_value,
                    'significant': significant,
                    'Larger if significant': larger,
                    'Smaller if significant': smaller
                })

        except Exception as error:
            for label in ['public vs non-public', 'mixed vs non-public', 'mixed vs public']:
                output_tables[model_name].append({
                    'Disease Mesh': disease_mesh,
                    'Comparison Model': label,
                    'coef': np.nan,
                    'p_value': np.nan,
                    'significant': False,
                    'Larger if significant': '',
                    'Smaller if significant': '',
                    'error': str(error)
                })

    return {model_name: pd.DataFrame(rows) for model_name, rows in output_tables.items()}

In [ ]:
def run_all_disease_models(
    df_list_citation,
    journal_var='decile',
    alpha=0.05
):

    all_results = {
        'c5_norm_OLS': [],
        'top1_pct_logit': [],
        'top10_pct_logit': []
    }

    for disease, df_disease in df_list_citation:

        try:
            result_dict = logistic_inst_type_compare(
                df=df_disease,
                disease_mesh=disease,
                journal_var=journal_var,
                alpha=alpha
            )

            print('in disease:', disease)


            for model_name, result_df in result_dict.items():

                if not result_df.empty:
                    all_results[model_name].append(result_df)

        except Exception as e:
            print(f'ERROR in disease {disease}: {e}')


            for model_name in all_results:

                error_rows = pd.DataFrame([
                    {
                        'Disease Mesh': disease,
                        'Comparison Model': 'public vs non-public',
                        'coef': np.nan,
                        'p_value': np.nan,
                        'significant': False,
                        'Larger if significant': '',
                        'Smaller if significant': '',
                        'error': str(e)
                    },
                    {
                        'Disease Mesh': disease,
                        'Comparison Model': 'mixed vs non-public',
                        'coef': np.nan,
                        'p_value': np.nan,
                        'significant': False,
                        'Larger if significant': '',
                        'Smaller if significant': '',
                        'error': str(e)
                    }
                ])

                all_results[model_name].append(error_rows)


    final_results = {}

    for model_name, result_list in all_results.items():

        if result_list:
            final_results[model_name] = pd.concat(
                result_list,
                ignore_index=True
            )
        else:
            final_results[model_name] = pd.DataFrame()

    return final_results

In [ ]:
df_results_q = run_all_disease_models(df_list_citation,journal_var='q_num')

In [ ]:
mesh_lookup = df_mesh_lvl1.rename(
    columns={
        "mesh_lvl1": "Disease Mesh",
        "category_name": "Disease_Name"
    }
)[["Disease Mesh", "Disease_Name"]]

In [ ]:
df_c5 = df_results_q["c5_norm_OLS"].merge(
    mesh_lookup,
    on="Disease Mesh",
    how="left"
)

df_top1 = df_results_q["top1_pct_logit"].merge(
    mesh_lookup,
    on="Disease Mesh",
    how="left"
)

df_top10 = df_results_q["top10_pct_logit"].merge(
    mesh_lookup,
    on="Disease Mesh",
    how="left"
)

In [ ]:
def reorder(df):
    first = [
        "Disease Mesh",
        "Disease_Name",
        "Comparison Model"
    ]
    return df[first + [c for c in df.columns if c not in first]]

df_c5 = reorder(df_c5)
df_top1 = reorder(df_top1)
df_top10 = reorder(df_top10)

## Policy citation matching and models

Identify papers cited by policy documents. For each cited paper, construct controls using publication period, journal, team-size group, document type, and disease-term similarity; use the resulting weights in logistic models.


In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_1, d_journal_2, on='PMID', how='inner')
df_cem_p = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype', '2017_sum', '2018_sum', '2019_sum', '2020_sum', '2021_sum', '2022_sum','total_sum',"team_size_group","document_type_group"]]

In [ ]:
df_cem_p['inst_type'] = df_cem_p['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
df_cem_p["year_group"] = pd.cut(
    df_cem_p["year"],
    bins=[2008, 2011, 2014, 2017, 2020, 2022],
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [ ]:
def match_one_stratum_cem_balance(
    stratum_df: pd.DataFrame,
    key_cols=("source_title", "year"),
    id_col="PMID",
    treat_col="treat",
    mesh_codes_col="mesh_codes",
    drop_if_no_overlap=True,
):
    """
    CEM-like within (source_title, year) exact stratum:
      - match each control to best Jaccard treatment(s) (ties allowed)
      - tie-splitting: a control matched to k tied treatments contributes 1/k to each
      - CEM-style balance: within each stratum, total treat weight = total control weight = 1

    Returns:
      pairs_df: columns [stratum_id, treat_pmid, control_pmid, sim, w_control_final]
      treat_df: columns [stratum_id, treat_pmid, w_treat_final]
    """
    tdf = stratum_df[stratum_df[treat_col] == 1].copy()
    cdf = stratum_df[stratum_df[treat_col] == 0].copy()

    if tdf.empty or cdf.empty:
        empty_pairs = pd.DataFrame(columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_control_final"])
        empty_treat = pd.DataFrame(columns=["stratum_id", "treat_pmid", "w_treat_final"])
        return empty_pairs, empty_treat

    # stratum_id
    stratum_vals = [str(stratum_df.iloc[0][k]) for k in key_cols]
    stratum_id = "||".join(stratum_vals)

    # treatments info
    t_ids = tdf[id_col].astype(str).tolist()
    t_mesh = tdf[mesh_codes_col].tolist()
    t_len = np.array([len(x) for x in t_mesh], dtype=np.int16)

    # Inverted index: code -> list of treatment indices
    inv = defaultdict(list)
    for ti, codes in enumerate(t_mesh):
        for code in codes:
            inv[code].append(ti)

    rows = []
    matched_control_ids = set()

    for _, crow in cdf.iterrows():
        c_id = str(crow[id_col])
        c_codes = crow[mesh_codes_col]
        c_len = len(c_codes)

        overlap = defaultdict(int)
        for code in c_codes:
            for ti in inv.get(code, []):
                overlap[ti] += 1

        if not overlap:
            if drop_if_no_overlap:
                continue
            else:

                best_tis = list(range(len(t_ids)))
                best_sim = 0.0
        else:
            best_sim = -1.0
            best_tis = []
            for ti, inter in overlap.items():
                union = c_len + int(t_len[ti]) - inter
                sim = inter / union if union > 0 else 0.0
                if sim > best_sim:
                    best_sim = sim
                    best_tis = [ti]
                elif sim == best_sim:
                    best_tis.append(ti)

        matched_control_ids.add(c_id)

        # tie-splitting within this control
        w_split = 1.0 / len(best_tis)
        for ti in best_tis:
            if overlap:
                inter = overlap[ti]
                union = c_len + int(t_len[ti]) - inter
                sim = inter / union if union > 0 else 0.0
            else:
                sim = best_sim  # 0
            rows.append([stratum_id, t_ids[ti], c_id, sim, w_split])

    pairs_df = pd.DataFrame(rows, columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_split"])


    if pairs_df.empty:
        empty_pairs = pd.DataFrame(columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_control_final"])
        empty_treat = pd.DataFrame(columns=["stratum_id", "treat_pmid", "w_treat_final"])
        return empty_pairs, empty_treat

    # --- CEM-style stratum balance weights ---
    n_treat = len(t_ids)
    #n_control = len(matched_control_ids)
    sum_split = pairs_df["w_split"].sum()


    w_treat_base = 1.0 / n_treat
    #w_control_base = 1.0 / n_control
    w_control_base = 1.0 / sum_split
    

    pairs_df["w_control_final"] = w_control_base * pairs_df["w_split"]
    pairs_df = pairs_df.drop(columns=["w_split"])


    treat_df = pd.DataFrame({
        "stratum_id": [stratum_id] * n_treat,
        "treat_pmid": t_ids,
        "w_treat_final": [w_treat_base] * n_treat
    })

    return pairs_df, treat_df

In [ ]:
df = df_cem_p.copy()
df["treat"] = (df["total_sum"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_p = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_p = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_p,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_p.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = [
    "PMID", "treat", "weight", "stratum_id",
    'matched_count','human','animal',
    "year", "year_group", "source_title",
    "team_size_group", "document_type_group",
    "inst_type",'funding_subtype', "mesh_lvl1", "total_sum"
]

keep_cols = [
    c for c in keep_cols
    if c in df_treat.columns and c in df_control_long.columns
]

df_matched_long_bal_p = pd.concat(
    [
        df_treat[keep_cols],
        df_control_long[keep_cols]
    ],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type)
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_p,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_p["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference="non-public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_p,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_p["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference="public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_p,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_p["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
def CEM_code(
    labeled_df : pd.DataFrame,
    treat_column,
    strata_cols = ["source_title", "year_group", "team_size_group", "document_type_group"]
):
    df = labeled_df.copy()
    df["treat"] = (df[treat_column] > 0).astype(int)
    #df["treat_p"] = (df["inst_type"] =='private').astype(int)
    df["mesh_len"] = df["mesh_lvl1"].apply(len)
    pairs_parts = []
    treat_parts = []

    for _, g in df.groupby(strata_cols, sort=False):
        p_df, t_df = match_one_stratum_cem_balance(
                g,
                key_cols=tuple(strata_cols),
                id_col="PMID",
                treat_col="treat",
                mesh_codes_col="mesh_lvl1",
                drop_if_no_overlap=True
        )
        if not p_df.empty:
            pairs_parts.append(p_df)
        if not t_df.empty:
            treat_parts.append(t_df)

    if not pairs_parts or not treat_parts:
        return df, pd.DataFrame()

    # print(f"Total matched pairs: {sum(len(p) for p in pairs_parts)}")
    # print(f"Total matched treatments: {sum(len(t) for t in treat_parts)}")
    df_pairs = pd.concat(pairs_parts, ignore_index=True)
    df_treat_w = pd.concat(treat_parts, ignore_index=True)

    df_treat = df[df["treat"] == 1].copy()
    df_treat["PMID"] = df_treat["PMID"].astype(str)
    df_treat["stratum_id"] = (
        df_treat[strata_cols]
        .astype(str)
        .agg("||".join, axis=1)
    )

    df_treat = df_treat.merge(
        df_treat_w, left_on=["stratum_id","PMID"], right_on=["stratum_id","treat_pmid"],
        how="inner", validate="many_to_one"
    )
    df_treat["weight"] = df_treat["w_treat_final"]

    df_control = df[df["treat"] == 0].copy()
    df_control["PMID"] = df_control["PMID"].astype(str)

    df_control_long = df_pairs.merge(
        df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
    )
    df_control_long["weight"] = df_control_long["w_control_final"]

    keep_cols = ["PMID","treat","weight","stratum_id","year",'year_group',"source_title","inst_type","mesh_lvl1",treat_column,'team_size_group', "document_type_group"]
    keep_cols = [c for c in keep_cols if c in df_treat.columns and c in df_control_long.columns]

    df_matched_long_bal = pd.concat(
        [df_treat[keep_cols], df_control_long[keep_cols]],
        ignore_index=True
    )

    return df, df_matched_long_bal


In [ ]:
df_list_policy = []

for disease in mesh_list:
    time_1 = time.time()
    df_disease_p = choose_disease(df_cem_p, disease)
    df_disease_p1, df_matched_p = CEM_code(df_disease_p, treat_column="total_sum", strata_cols=["source_title", "year_group","team_size_group","document_type_group"])
    df_list_policy.append([disease, df_disease_p1, df_matched_p])
    time_2 = time.time()
    print(f"Finished processing {disease} in {time_2 - time_1:.2f} seconds.")

In [ ]:
def get_disease_data(df_list_policy, alpha=0.05):
    all_results = []

    inst_types = ["total", "public", "non-public", "mixed"]

    for disease, df_disease, df_matched in df_list_policy:

        for inst in inst_types:

            # ====== df_disease ======
            if inst == "total":
                d_sub = df_disease
            else:
                d_sub = df_disease[df_disease["inst_type"] == inst]

            d_t1 = d_sub[d_sub["treat"] == 1]["PMID"].nunique()
            d_t0 = d_sub[d_sub["treat"] == 0]["PMID"].nunique()

            # ====== df_matched ======
            if inst == "total":
                m_sub = df_matched
            else:
                m_sub = df_matched[df_matched["inst_type"] == inst]

            m_t1 = m_sub[m_sub["treat"] == 1]["PMID"].nunique()
            m_t0 = m_sub[m_sub["treat"] == 0]["PMID"].nunique()


            row = pd.DataFrame([{
                "disease": disease,
                "inst_type": inst,

                "number_treatment_before_CEM": d_t1,
                "number_control_before_CEM": d_t0,

                "number_treatment_after_CEM": m_t1,
                "number_control_after_CEM": m_t0
            }])

            all_results.append(row)

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    else:
        return pd.DataFrame()

In [ ]:
import pandas as pd
import statsmodels.api as sm
import patsy

def logistic_inst_type_compare(df_weight,disease_name, alpha=0.05):
    """
    Compare inst_type effects using logistic regression.
    
    Input:
        df_weight: dataframe with treatment and weight information
        disease_name: label for the disease / MeSH code
        alpha: significance threshold, default 0.05
    
    Output:
        dataframe showing whether each pair is significant and which group is larger
    """

    df_model = df_weight[['treat', 'weight', 'inst_type','year_group','team_size_group','document_type_group']].copy()
    df_model = df_model.dropna(subset=['treat', 'weight', 'inst_type','year_group','team_size_group','document_type_group'])


    df_model['inst_type'] = df_model['inst_type'].astype(str)
    df_model['weight'] = df_model['weight'].astype(float)

    results = []

    def run_model_with_baseline(baseline):
        formula = f"treat ~ C(inst_type, Treatment(reference='{baseline}'))+ C(year_group)+C(team_size_group)+C(document_type_group)"

        y, X = patsy.dmatrices(
            formula,
            df_model,
            return_type="dataframe"
        )

        model = sm.GLM(
            y,
            X,
            family=sm.families.Binomial(),
            freq_weights=df_model.loc[y.index, 'weight']
        )

        res = model.fit()
        return res

    def extract_comparison(res, baseline):
        for term in res.params.index:
            if "C(inst_type" not in term:
                continue



            compared_group = term.split("[T.")[-1].replace("]", "")

            coef = res.params[term]
            pval = res.pvalues[term]

            if pval < alpha:
                significant = True
                if coef > 0:
                    larger_group = compared_group
                    smaller_group = baseline
                else:
                    larger_group = baseline
                    smaller_group = compared_group
            else:
                significant = False
                larger_group = None
                smaller_group = None

            results.append({
                "disease": disease_name,
                "comparison": f"{compared_group} vs {baseline}",
                "coef": coef,
                "p_value": pval,
                "significant": significant,
                "larger_if_significant": larger_group,
                "smaller_if_significant": smaller_group
            })


    res_non_public = run_model_with_baseline("non-public")
    extract_comparison(res_non_public, "non-public")


    res_public = run_model_with_baseline("public")
    extract_comparison(res_public, "public")

    result_df = pd.DataFrame(results)


    result_df["pair_key"] = result_df["comparison"].apply(
        lambda x: tuple(sorted(x.split(" vs ")))
    )

    result_df = result_df.drop_duplicates(subset=["pair_key"]).drop(columns=["pair_key"])

    return result_df

In [ ]:
def run_all_disease_models(df_list_policy, alpha=0.05):
    all_results = []

    for disease, df_disease, df_matched in df_list_policy:
        try:
            result = logistic_inst_type_compare(
                df_weight=df_matched,
                disease_name=disease,
                alpha=alpha
            )

            if not result.empty:
                all_results.append(result)

        except Exception as e:
            all_results.append(pd.DataFrame([{
                "disease": disease,
                "comparison": None,
                "coef": None,
                "p_value": None,
                "significant": None,
                "relation": f"ERROR: {e}"
            }]))

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    else:
        return pd.DataFrame()

In [ ]:
policy_result_all = run_all_disease_models(df_list_policy)
policy_result_all

## Patent citation matching and models

Link PKG patent-to-paper records, identify cited papers, construct matched controls using the same design, and specify the patent citation models.


In [ ]:
df_pp = pd.read_csv("PKG\C16_Link_Patents_Papers.tsv", encoding="utf-8", sep="\t")
df_ppc = df_pp.drop_duplicates(['PMID','PatentId'])[['PMID','PatentId']].groupby('PMID').count().reset_index()
df_final_pattern = df_1.merge(df_ppc, on="PMID", how="left")
df_final_pattern["PatentId"] = df_final_pattern["PatentId"].fillna(0).astype(int)
df_final_pattern["has_patent"] = (df_final_pattern["PatentId"] > 0).astype(int)

In [ ]:
df_final_pattern['inst_type'] = df_final_pattern['inst_type'].replace({
        'private': 'non-public',
        'public+private': 'mixed'
    })

In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_final_pattern, d_journal_2, on='PMID', how='inner')
df_cem_pp = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype',"has_patent","team_size_group","document_type_group"]]

In [ ]:
df_cem_pp["year_group"] = pd.cut(
    df_cem_pp["year"],
    bins=[2008, 2011, 2014, 2017, 2020, 2022],
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [ ]:
df_cem_pp['has_patent'].value_counts()

In [ ]:
def match_one_stratum_cem_balance(
    stratum_df: pd.DataFrame,
    key_cols=("source_title", "year"),
    id_col="PMID",
    treat_col="treat",
    mesh_codes_col="mesh_codes",
    drop_if_no_overlap=True,
):
    """
    CEM-like within (source_title, year) exact stratum:
      - match each control to best Jaccard treatment(s) (ties allowed)
      - tie-splitting: a control matched to k tied treatments contributes 1/k to each
      - CEM-style balance: within each stratum, total treat weight = total control weight = 1

    Returns:
      pairs_df: columns [stratum_id, treat_pmid, control_pmid, sim, w_control_final]
      treat_df: columns [stratum_id, treat_pmid, w_treat_final]
    """
    tdf = stratum_df[stratum_df[treat_col] == 1].copy()
    cdf = stratum_df[stratum_df[treat_col] == 0].copy()

    if tdf.empty or cdf.empty:
        empty_pairs = pd.DataFrame(columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_control_final"])
        empty_treat = pd.DataFrame(columns=["stratum_id", "treat_pmid", "w_treat_final"])
        return empty_pairs, empty_treat

    # stratum_id
    stratum_vals = [str(stratum_df.iloc[0][k]) for k in key_cols]
    stratum_id = "||".join(stratum_vals)

    # treatments info
    t_ids = tdf[id_col].astype(str).tolist()
    t_mesh = tdf[mesh_codes_col].tolist()
    t_len = np.array([len(x) for x in t_mesh], dtype=np.int16)

    # Inverted index: code -> list of treatment indices
    inv = defaultdict(list)
    for ti, codes in enumerate(t_mesh):
        for code in codes:
            inv[code].append(ti)

    rows = []
    matched_control_ids = set()

    for _, crow in cdf.iterrows():
        c_id = str(crow[id_col])
        c_codes = crow[mesh_codes_col]
        c_len = len(c_codes)

        overlap = defaultdict(int)
        for code in c_codes:
            for ti in inv.get(code, []):
                overlap[ti] += 1

        if not overlap:
            if drop_if_no_overlap:
                continue
            else:

                best_tis = list(range(len(t_ids)))
                best_sim = 0.0
        else:
            best_sim = -1.0
            best_tis = []
            for ti, inter in overlap.items():
                union = c_len + int(t_len[ti]) - inter
                sim = inter / union if union > 0 else 0.0
                if sim > best_sim:
                    best_sim = sim
                    best_tis = [ti]
                elif sim == best_sim:
                    best_tis.append(ti)

        matched_control_ids.add(c_id)

        # tie-splitting within this control
        w_split = 1.0 / len(best_tis)
        for ti in best_tis:
            if overlap:
                inter = overlap[ti]
                union = c_len + int(t_len[ti]) - inter
                sim = inter / union if union > 0 else 0.0
            else:
                sim = best_sim  # 0
            rows.append([stratum_id, t_ids[ti], c_id, sim, w_split])

    pairs_df = pd.DataFrame(rows, columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_split"])


    if pairs_df.empty:
        empty_pairs = pd.DataFrame(columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_control_final"])
        empty_treat = pd.DataFrame(columns=["stratum_id", "treat_pmid", "w_treat_final"])
        return empty_pairs, empty_treat

    # --- CEM-style stratum balance weights ---
    n_treat = len(t_ids)
    #n_control = len(matched_control_ids)
    sum_split = pairs_df["w_split"].sum()


    w_treat_base = 1.0 / n_treat
    #w_control_base = 1.0 / n_control
    w_control_base = 1.0 / sum_split
    

    pairs_df["w_control_final"] = w_control_base * pairs_df["w_split"]
    pairs_df = pairs_df.drop(columns=["w_split"])


    treat_df = pd.DataFrame({
        "stratum_id": [stratum_id] * n_treat,
        "treat_pmid": t_ids,
        "w_treat_final": [w_treat_base] * n_treat
    })

    return pairs_df, treat_df

In [ ]:
df = df_cem_pp.copy()
df["treat"] = (df["has_patent"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_pp = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_pp = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_pp,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_pp.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = ["PMID","treat","weight","stratum_id",'matched_count','human','animal',"year",'year_group',"source_title","inst_type",'funding_subtype',"mesh_lvl1","has_patent","team_size_group","document_type_group"]
keep_cols = [c for c in keep_cols if c in df_treat.columns and c in df_control_long.columns]

df_matched_long_bal_pp = pd.concat(
    [df_treat[keep_cols], df_control_long[keep_cols]],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type)
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_pp,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_pp["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference="non-public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_pp,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_pp["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference="public"))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_pp,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_pp["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
df_list_patent = []

for disease in mesh_list:
    time_1 = time.time()
    df_disease_pp = choose_disease(df_cem_pp, disease)
    df_disease_pp1, df_matched_pp = CEM_code(df_disease_pp, treat_column="has_patent", strata_cols=["source_title", "year_group","team_size_group","document_type_group"])
    df_list_patent.append([disease, df_disease_pp1, df_matched_pp])
    time_2 = time.time()
    print(f"Finished processing {disease} in {time_2 - time_1:.2f} seconds.")

In [ ]:
patent_result_count = get_disease_data(df_list_patent).merge(
    df_mesh_lvl1[['mesh_lvl1', 'category_name']].rename(columns={'category_name':'disease_name'}),
    left_on='disease',
    right_on='mesh_lvl1',
    how='left'
)[['disease', 'disease_name', 'inst_type', 'number_treatment_after_CEM', 'number_control_after_CEM', 'number_treatment_before_CEM', 'number_control_before_CEM']]
patent_result_count.to_excel("a5_patent_disease_result_count.xlsx", index=False)
patent_result_count

In [ ]:
patent_result_all = run_all_disease_models(df_list_patent)
patent_result_all

## Clinical-trial citation matching and models

Link PKG clinical-trial-to-paper records, identify cited papers, construct matched controls, and specify the clinical-trial citation models.


In [ ]:
d_journal = pd.read_json('a3_data_journal.json', lines=True)
d_journal_2 = d_journal[['pmid_uscfg_list','source_title','conferences']].rename(columns={'pmid_uscfg_list':'PMID'})
df_1j = pd.merge(df_final_clinical, d_journal_2, on='PMID', how='inner')
df_cem_c = df_1j[['PMID','year','source_title','matched_count','human','animal','mesh_lvl1','inst_type','funding_subtype',"has_CT","team_size_group","document_type_group"]]

In [ ]:
df_cem_c["year_group"] = pd.cut(
    df_cem_c["year"],
    bins=[2004, 2007, 2010, 2013, 2016, 2019, 2022],
    labels=[1, 2, 3, 4, 5, 6]
).astype(int)

In [ ]:
def match_one_stratum_cem_balance(
    stratum_df: pd.DataFrame,
    key_cols=("source_title", "year"),
    id_col="PMID",
    treat_col="treat",
    mesh_codes_col="mesh_codes",
    drop_if_no_overlap=True,
):
    """
    CEM-like within (source_title, year) exact stratum:
      - match each control to best Jaccard treatment(s) (ties allowed)
      - tie-splitting: a control matched to k tied treatments contributes 1/k to each
      - CEM-style balance: within each stratum, total treat weight = total control weight = 1

    Returns:
      pairs_df: columns [stratum_id, treat_pmid, control_pmid, sim, w_control_final]
      treat_df: columns [stratum_id, treat_pmid, w_treat_final]
    """
    tdf = stratum_df[stratum_df[treat_col] == 1].copy()
    cdf = stratum_df[stratum_df[treat_col] == 0].copy()

    if tdf.empty or cdf.empty:
        empty_pairs = pd.DataFrame(columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_control_final"])
        empty_treat = pd.DataFrame(columns=["stratum_id", "treat_pmid", "w_treat_final"])
        return empty_pairs, empty_treat

    # stratum_id
    stratum_vals = [str(stratum_df.iloc[0][k]) for k in key_cols]
    stratum_id = "||".join(stratum_vals)

    # treatments info
    t_ids = tdf[id_col].astype(str).tolist()
    t_mesh = tdf[mesh_codes_col].tolist()
    t_len = np.array([len(x) for x in t_mesh], dtype=np.int16)

    # Inverted index: code -> list of treatment indices
    inv = defaultdict(list)
    for ti, codes in enumerate(t_mesh):
        for code in codes:
            inv[code].append(ti)

    rows = []
    matched_control_ids = set()

    for _, crow in cdf.iterrows():
        c_id = str(crow[id_col])
        c_codes = crow[mesh_codes_col]
        c_len = len(c_codes)

        overlap = defaultdict(int)
        for code in c_codes:
            for ti in inv.get(code, []):
                overlap[ti] += 1

        if not overlap:
            if drop_if_no_overlap:
                continue
            else:

                best_tis = list(range(len(t_ids)))
                best_sim = 0.0
        else:
            best_sim = -1.0
            best_tis = []
            for ti, inter in overlap.items():
                union = c_len + int(t_len[ti]) - inter
                sim = inter / union if union > 0 else 0.0
                if sim > best_sim:
                    best_sim = sim
                    best_tis = [ti]
                elif sim == best_sim:
                    best_tis.append(ti)

        matched_control_ids.add(c_id)

        # tie-splitting within this control
        w_split = 1.0 / len(best_tis)
        for ti in best_tis:
            if overlap:
                inter = overlap[ti]
                union = c_len + int(t_len[ti]) - inter
                sim = inter / union if union > 0 else 0.0
            else:
                sim = best_sim  # 0
            rows.append([stratum_id, t_ids[ti], c_id, sim, w_split])

    pairs_df = pd.DataFrame(rows, columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_split"])


    if pairs_df.empty:
        empty_pairs = pd.DataFrame(columns=["stratum_id", "treat_pmid", "control_pmid", "sim", "w_control_final"])
        empty_treat = pd.DataFrame(columns=["stratum_id", "treat_pmid", "w_treat_final"])
        return empty_pairs, empty_treat

    # --- CEM-style stratum balance weights ---
    n_treat = len(t_ids)
    #n_control = len(matched_control_ids)
    sum_split = pairs_df["w_split"].sum()


    w_treat_base = 1.0 / n_treat
    #w_control_base = 1.0 / n_control
    w_control_base = 1.0 / sum_split
    

    pairs_df["w_control_final"] = w_control_base * pairs_df["w_split"]
    pairs_df = pairs_df.drop(columns=["w_split"])


    treat_df = pd.DataFrame({
        "stratum_id": [stratum_id] * n_treat,
        "treat_pmid": t_ids,
        "w_treat_final": [w_treat_base] * n_treat
    })

    return pairs_df, treat_df

In [ ]:
df = df_cem_c.copy()
df["treat"] = (df["has_CT"] > 0).astype(int)
df["mesh_len"] = df["mesh_lvl1"].apply(len)

strata_cols = [
    "year_group",
    "source_title",
    "team_size_group",
    "document_type_group"
]

pairs_parts = []
treat_parts = []

for _, g in df.groupby(strata_cols, sort=False, observed=True):
    p_df, t_df = match_one_stratum_cem_balance(
        g,
        key_cols=tuple(strata_cols),
        id_col="PMID",
        treat_col="treat",
        mesh_codes_col="mesh_lvl1",
        drop_if_no_overlap=True
    )

    if not p_df.empty:
        pairs_parts.append(p_df)

    if not t_df.empty:
        treat_parts.append(t_df)

df_pairs_c = pd.concat(pairs_parts, ignore_index=True)
df_treat_w_c = pd.concat(treat_parts, ignore_index=True)

In [ ]:
df_treat = df[df["treat"] == 1].copy()
df_treat["PMID"] = df_treat["PMID"].astype(str)

df_treat["stratum_id"] = (
    df_treat[strata_cols]
    .astype(str)
    .agg("||".join, axis=1)
)

df_treat = df_treat.merge(
    df_treat_w_c,
    left_on=["stratum_id", "PMID"],
    right_on=["stratum_id", "treat_pmid"],
    how="inner",
    validate="many_to_one"
)

df_treat["weight"] = df_treat["w_treat_final"]

In [ ]:
df_control = df[df["treat"] == 0].copy()
df_control["PMID"] = df_control["PMID"].astype(str)

df_control_long = df_pairs_c.merge(
    df_control, left_on="control_pmid", right_on="PMID", how="left", validate="many_to_one"
)
df_control_long["weight"] = df_control_long["w_control_final"]

In [ ]:
keep_cols = ["PMID","treat","weight","stratum_id",'matched_count','human','animal',"year",'year_group',"source_title","inst_type",'funding_subtype',"mesh_lvl1","has_CT","team_size_group","document_type_group"]
keep_cols = [c for c in keep_cols if c in df_treat.columns and c in df_control_long.columns]

df_matched_long_bal_c = pd.concat(
    [df_treat[keep_cols], df_control_long[keep_cols]],
    ignore_index=True
)

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type)
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_c,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_c["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference='non-public'))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_c,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_c["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
import statsmodels.api as sm
import patsy

formula = """
treat ~ C(inst_type, Treatment(reference='public'))
      + C(year_group)
      + C(team_size_group)
      + C(document_type_group)
"""

y, X = patsy.dmatrices(
    formula,
    df_matched_long_bal_c,
    return_type="dataframe"
)

model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=df_matched_long_bal_c["weight"].astype(float)
)

res = model.fit()
print(res.summary())

In [ ]:
df_list_clinical = []

for disease in mesh_list:
    time_1 = time.time()
    df_disease_c = choose_disease(df_cem_c, disease)
    df_disease_c1, df_matched = CEM_code(df_disease_c, treat_column="has_CT", strata_cols=["source_title", "year_group","team_size_group","document_type_group"])
    if df_matched.empty:
        print(f"No matched sample for {disease}")
    else:
        df_list_clinical.append([disease, df_disease_c1, df_matched])
    time_2 = time.time()
    print(f"Finished processing {disease} in {time_2 - time_1:.2f} seconds.")